# N124 · 位运算与整数表示

**目标：** 在明确位宽下操作二进制并区分Python整数语义。

**先修：** N002, N005, N013。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 与或异或非；移位；补码；lowbit；清最低1；popcount；掩码。

## 从问题到算法

位运算在二进制表示上操作。n&(n−1)删除最低位的1，因此循环次数等于置位数。Python整数没有固定32位溢出，位反转题必须显式截取低32位；负整数的无限符号扩展不能直接套“不断清除1直到零”的循环。二进制加法逐位维护carry。

## 最小实现

**本章接口：** `popcount(n)`、`reverse_bits32(n)`、`add_binary(a, b)`

In [1]:
def popcount(n):
    if n<0: raise ValueError('popcount requires nonnegative integer')
    count=0
    while n: n&=n-1; count+=1
    return count

def reverse_bits32(n):
    n&=(1<<32)-1; out=0
    for _ in range(32): out=(out<<1)|(n&1); n>>=1
    return out

def add_binary(a,b):
    if any(c not in '01' for c in a+b): raise ValueError('binary strings required')
    i,j=len(a)-1,len(b)-1; carry=0; out=[]
    while i>=0 or j>=0 or carry:
        if i>=0: carry+=ord(a[i])-48; i-=1
        if j>=0: carry+=ord(b[j])-48; j-=1
        out.append(str(carry&1)); carry>>=1
    return ''.join(reversed(out)).lstrip('0') or '0'

## 正确性、前提与复杂度

n−1把最低1变为0，并把其右侧零变成1；与原数相与正好只删除那一位。反转循环每步把原数最低位接到输出末尾，32步后各位位置映射i→31−i。加法carry记录来自较低位的进位，保持部分和一致。

**代价：** popcount执行O(置位数)次整数位操作；任意精度整数的每次位操作成本随位宽增长。32位反转固定O(1)；字符串加法O(max(m,n))时间和输出空间。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

n=44; rows=[]
while n:
    nxt=n&(n-1); rows.append((n,format(n,'08b'),format(nxt,'08b'))); n=nxt
show_table(['十进制','二进制','清除最低1'],rows)

十进制,二进制,清除最低1
44,00101100,00101000
40,00101000,00100000
32,00100000,00000000


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert popcount(0)==0 and popcount(7)==3
assert add_binary('11','1')=='100' and add_binary('000','0')=='0'
for n in range(1000):
    assert popcount(n)==n.bit_count()
    assert reverse_bits32(reverse_bits32(n))==n
assert reverse_bits32(-1)==(1<<32)-1
for a in range(64):
    for b in range(64): assert add_binary(format(a,'b'),format(b,'b'))==format(a+b,'b')
try: popcount(-1)
except ValueError: pass
else: raise AssertionError('negative domain must be rejected')
print('N124: 所有本章断言通过')

N124: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 说明Python的~x不是有限位按位翻转结果。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 解释大整数位操作不总是常数成本。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 191. Number of 1 Bits（canonical）
- 190. Reverse Bits（canonical）
- 338. Counting Bits（canonical）
- 67. Add Binary（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。